# P62 — La IA y el benchmark del todo en el mundo entero

## 1. Título y paper

**Paper:** *AI and the Everything in the Whole Wide World Benchmark*  
**Autoría:** Inioluwa Deborah Raji, Emily M. Bender, Amandalynne Paullada, Emily Denton, Alex Hanna  
**Año y venue:** 2021 · NeurIPS 2021 · Datasets and Benchmarks Track · arXiv:2111.15366  
**Nivel:** L3 · **Motor:** `benchmark_validez`  
**Ficha completa:** [`P62_benchmark_validez`](../../papers/foundational/P62_benchmark_validez/README.md)

**Hito:** Traslada al campo el concepto de validez de constructo: un número alto no prueba la capacidad que el benchmark dice medir.

- [arXiv:2111.15366](https://arxiv.org/abs/2111.15366)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los benchmarks se presentaban como pruebas de capacidades generales —«comprensión», «razonamiento»— cuando sus ítems cubren una porción estrecha y a menudo admiten atajos.
2. Ejecutar una implementación mínima de la propuesta: Evaluar los benchmarks como instrumentos de medida: preguntar qué constructo dicen medir, qué cubren realmente sus ítems y qué estrategias los superan sin la capacidad.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Torralba y Efros (2011), sesgo de los conjuntos de datos
- P51


## 4. Intuición

Un benchmark que se llama «comprensión» promete medir comprensión. Pero sus ítems miden lo que miden, y si existe un atajo —responder siempre la opción más larga— el ranking mide el atajo. El problema no es la métrica: es la distancia entre lo medido y lo afirmado.


## 5. Concepto mínimo

```text
Validez de constructo:  ¿la tarea medida ES la capacidad nombrada?

    capacidad declarada : «comprensión de lenguaje natural»
    subhabilidades      : 6 declaradas
    subhabilidades medidas: 2
    estrategia sin comprensión: 11/12 aciertos

Un número alto es compatible con no tener la capacidad.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('benchmark_validez', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta una regla que ignora el contenido del ítem?
2. ¿Cuántas de las seis subhabilidades declaradas se evalúan?
3. ¿Qué mide entonces el ranking?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('benchmark_validez', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('benchmark_validez', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La regla «responder siempre la opción más larga» acierta 11 de 12, frente a los 3 de 12 del azar. Y de las seis subhabilidades declaradas solo se evalúan dos. El ranking mide el atajo y la etiqueta promete un constructo que los ítems no cubren.


## 10. Comentario pedagógico

Esta es la ficha que convierte el escepticismo en método. Antes de creer una tabla comparativa: mirar los ítems, buscar el atajo y comprobar la cobertura. [SWE-bench](../../papers/foundational/P51_swebench/README.md) es interesante justamente porque reduce el hueco —los tests del repositorio son difíciles de fingir—, no porque sea un benchmark más grande.


## 11. Error o anti-patrón deliberado

Anti-patrón: comparar dos modelos por su puntuación sin haber mirado un solo ítem.


In [ ]:
print('Un ranking es una medida CON un instrumento, no una propiedad del modelo.')
print('Si el instrumento admite atajos, la puntuacion mide el atajo.')
print('Y si el conjunto de test se filtro al entrenamiento, no mide nada.')

## 12. Corrección

El procedimiento mínimo antes de citar un número:


In [ ]:
r = run_paper_lab('benchmark_validez', seed=7)['result']
print('cobertura declarada vs real :', r['cobertura'])
print('atajo sin capacidad         :', r['modelo_atajo']['aciertos'])
print('azar                        :', r['modelo_al_azar']['aciertos_esperados'])
print('items a inspeccionar a mano :', r['items_a_inspeccionar_a_mano'])

## 13. Desafío guiado

Con la salida del motor, calcula cuánta ventaja saca el atajo sobre el azar y argumenta qué conclusión se puede y no se puede sacar de una exactitud del 91,7 %.


In [ ]:
r = run_paper_lab('benchmark_validez', seed=3)['result']
show(r)

## 14. Desafío autónomo

Elige un benchmark real que se use en tu área, lee veinte de sus ítems y responde por escrito: qué constructo declara, qué subhabilidades cubre y qué atajo se te ocurre. Después busca si alguien ya lo publicó.


## 15. Evidencia de aprendizaje

Guarda la tabla de cobertura y la puntuación del atajo, junto con tu procedimiento de tres pasos para auditar un benchmark.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P62_benchmark_validez/README.md) · evaluación formal: [`assessments/papers/P62_benchmark_validez.md`](../../assessments/papers/P62_benchmark_validez.md)


## 16. Cierre

Ya sabemos desconfiar del instrumento. Queda desconfiar del procedimiento: qué hace falta para que otra persona obtenga el mismo número.


## 17. Conexión con el siguiente hito

- P51

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
